# Animation: the rotation axis and the GRACE load

Left: the rotation axis $\Omega+\boldsymbol\omega$ on a globe with the model's coastlines, dynamic (solid) and static (dashed). The real tilt is tens
of milliarcseconds, so the tilt angle is multiplied by a large factor; the shape of the motion is exact. The trail shows the last two years. Middle: the direct load each month
(linear colour scale within ±50 mm, logarithmic beyond). Right: the pole paths from above, at true scale. Frames are the observed months.

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

# solver_dynamic and examples_dynamic_helper live one directory up, in new-code/
sys.path.append("..")

import solver_dynamic as sd
import examples_dynamic_helper as eh
import grace_data_helper as gh
import grace_plot_helper as ph

/space/dah94/rotation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#=======================================================================
# Parameters
#=======================================================================

# The run to plot (see grace_rotation_run.py): forcing, degree and quality factor
forcing = "land-ocean"
lmax = 64
quality_factor = "100"

# Largest tilt of the dynamic axis in the drawing (degrees), and the length of its trail (months)
max_tilt_degrees = 20.0
trail_months = 24

# The load's colour scale is linear within +-load_linear_mm and logarithmic beyond
load_linear_mm = 50.0

frames_per_second = 8
dpi = 80
gif_path = f"animations/grace_pole_{forcing}.gif"

In [3]:
#=======================================================================
# Earth model, the run and the direct load
#=======================================================================

state = sd.make_state(lmax)
model = state.model
constants = sd.rotation_constants(model)

run = gh.load_run(forcing, lmax, quality_factor)
time_days, dates, load_coeffs = gh.direct_load_coefficients(forcing, lmax)
assert np.allclose(run["month_days"], time_days), "Plot a run that covers every month"

dynamic_mas = sd.omega_to_mas(run["dynamic_omega"], constants)
static_mas = sd.omega_to_mas(run["static_omega"], constants)

# The dynamic pole at every time step, and the step of each month (the frames)
path_mas = run["pole_mas"]
frames = ph.frame_index(run["t_days"], run["month_days"])
print(f"{len(dates)} months, {dates[0]} to {dates[-1]}")

259 months, 2002-04-17 to 2026-07-16


In [4]:
#=======================================================================
# Exaggerated axis directions, the globe and the load
#=======================================================================

max_tilt = np.linalg.norm(run["dynamic_omega"], axis=1).max() / constants["omega"]
exaggeration = np.radians(max_tilt_degrees) / max_tilt
print(f"tilt exaggeration factor {exaggeration:.3g}")

axis = eh.rotation_axis_direction(run["dynamic_omega"], constants["omega"], exaggeration)
axis_static = eh.rotation_axis_direction(run["static_omega"], constants["omega"], exaggeration)
axis_length = 1.5

# Globe: a light wireframe and the coastlines of the ocean function
lon_grid, lat_grid = np.meshgrid(np.linspace(0, 360, 37), np.linspace(-90, 90, 19))
sphere_x, sphere_y, sphere_z = eh.lonlat_to_xyz(lon_grid, lat_grid)
coastlines = [eh.lonlat_to_xyz(segment[:, 0], segment[:, 1]) for segment in eh.coastline_segments(state)]

# View from the side the pole drifts towards, raised above the equator
mean_azimuth = np.degrees(np.arctan2(dynamic_mas[:, 1].mean(), dynamic_mas[:, 0].mean()))

load_mm = ph.fields_from_coefficients(model, load_coeffs)
load_norm = ph.load_norm(load_mm, linear_mm=load_linear_mm)

tilt exaggeration factor 3.91e+05


In [5]:
#=======================================================================
# Animation: 3D axis on the globe, the load, and the pole path from above
#=======================================================================

fig = plt.figure(figsize=(16, 5.2))
grid = fig.add_gridspec(1, 3, width_ratios=[0.9, 1.25, 0.75], wspace=0.15, left=0.0, right=0.98, top=0.92, bottom=0.15)
ax_3d = fig.add_subplot(grid[0], projection="3d")
ax_map = fig.add_subplot(grid[1], projection=ccrs.Robinson())
ax_top = fig.add_subplot(grid[2])

# --- Globe and axes ---
ax_3d.plot_wireframe(sphere_x, sphere_y, sphere_z, color="lightgrey", lw=0.3)
for x, y, z in coastlines:
    ax_3d.plot(x, y, z, color="tab:green", lw=0.6, alpha=0.7)
ax_3d.plot([0, 0], [0, 0], [-axis_length, axis_length], color="grey", lw=0.8, ls=":")
static_line, = ax_3d.plot([], [], [], "r--", lw=1.5, label="static axis")
axis_line, = ax_3d.plot([], [], [], "k", lw=2.5, label="dynamic axis")
tip_marker, = ax_3d.plot([], [], [], "ko", ms=5)
tip_trace, = ax_3d.plot([], [], [], "k", lw=0.8)
ax_3d.set_xlim(-1.1, 1.1)
ax_3d.set_ylim(-1.1, 1.1)
ax_3d.set_zlim(-1.1, 1.6)
ax_3d.set_box_aspect((1, 1, 2.7 / 2.2), zoom=1.6)
ax_3d.view_init(elev=35, azim=mean_azimuth)
ax_3d.set_axis_off()
ax_3d.legend(loc="upper left")
title = ax_3d.set_title("")

# --- Load map ---
mesh = ax_map.pcolormesh(state.lons(), state.lats(), load_mm[0], transform=ccrs.PlateCarree(), cmap="BrBG",
                         norm=load_norm, shading="auto")
ax_map.contour(state.lons(), state.lats(), state.ocean_function.data, levels=[0.5], colors="k", linewidths=0.5,
               transform=ccrs.PlateCarree())
ax_map.set_global()
load_ticks = ph.colorbar_ticks(load_norm)
colorbar = fig.colorbar(mesh, ax=ax_map, orientation="horizontal", pad=0.04, fraction=0.06, shrink=0.85, extend="both",
                        label="Direct load (mm water equivalent)", ticks=load_ticks)
colorbar.ax.set_xticklabels([f"{tick:g}" for tick in load_ticks])
map_title = ax_map.set_title("")

# --- Pole paths from above ---
ax_top.plot(static_mas[:, 0], static_mas[:, 1], "r--", lw=0.6, alpha=0.7, label="static")
ax_top.plot(path_mas[:, 0], path_mas[:, 1], color="lightgrey", lw=0.6)
top_trace, = ax_top.plot([], [], "k", lw=1.2, label="dynamic")
top_marker, = ax_top.plot([], [], "ko")
ax_top.set_xlabel(r"$\omega_1/\Omega$ (mas)")
ax_top.set_ylabel(r"$\omega_2/\Omega$ (mas)")
ax_top.set_aspect("equal")
ax_top.grid(True, alpha=0.5)
ax_top.legend(loc="upper left", fontsize=9)
ax_top.set_title("Pole paths from above (true scale)")


def update(n):
    tip = axis_length * axis[n]
    tip_static = axis_length * axis_static[n]
    axis_line.set_data_3d([0, tip[0]], [0, tip[1]], [0, tip[2]])
    static_line.set_data_3d([0, tip_static[0]], [0, tip_static[1]], [0, tip_static[2]])
    tip_marker.set_data_3d([tip[0]], [tip[1]], [tip[2]])
    trace = axis_length * axis[max(0, n - trail_months): n + 1]
    tip_trace.set_data_3d(trace[:, 0], trace[:, 1], trace[:, 2])
    mesh.set_array(load_mm[n].ravel())
    map_title.set_text(f"Direct load, {str(dates[n])[:7]}")
    top_trace.set_data(path_mas[: frames[n] + 1, 0], path_mas[: frames[n] + 1, 1])
    top_marker.set_data([path_mas[frames[n], 0]], [path_mas[frames[n], 1]])
    title.set_text(f"{str(dates[n])[:7]}; tilt x {exaggeration:.2g}")
    return axis_line, static_line, tip_marker, tip_trace, mesh, map_title, top_trace, top_marker, title


animation = FuncAnimation(fig, update, frames=len(dates), interval=1000 / frames_per_second, blit=False)
plt.close(fig)

os.makedirs(os.path.dirname(gif_path), exist_ok=True)
animation.save(gif_path, writer=PillowWriter(fps=frames_per_second), dpi=dpi)
print(f"saved {gif_path}")
Image(url=gif_path)

saved animations/grace_pole_land-ocean.gif
